In [1]:
import sys
print(sys.executable)

C:\Florence's\Work Environtment\Magang - Imigrasi\Facial Scores\Trial_1\.venv\Scripts\python.exe


In [3]:
import pandas as pd
import numpy as np
import os
import cv2
from tqdm.notebook import tqdm
from keras.applications import MobileNetV2
from keras.models import Sequential, Model
from keras.layers import Dense, Dropout, Flatten, GlobalAveragePooling2D, Input
from keras.callbacks import ModelCheckpoint, ReduceLROnPlateau, EarlyStopping
from keras.utils import to_categorical
from keras_preprocessing.image import ImageDataGenerator, load_img
from sklearn.utils.class_weight import compute_class_weight
from sklearn.preprocessing import LabelEncoder

In [4]:
TRAIN_DIR = 'images/train'
TEST_DIR = 'images/test'

def createdataframe(dir):
    image_paths = []
    labels = []
    for label in os.listdir(dir):
        for imagename in os.listdir(os.path.join(dir, label)):
            image_paths.append(os.path.join(dir, label, imagename))
            labels.append(label)
        print(label, "completed")
    return image_paths, labels

train = pd.DataFrame()
train['image'], train['label'] = createdataframe(TRAIN_DIR)

test = pd.DataFrame()
test['image'], test['label'] = createdataframe(TEST_DIR)

angry completed
disgust completed
fear completed
happy completed
neutral completed
sad completed
surprise completed
angry completed
disgust completed
fear completed
happy completed
neutral completed
sad completed
surprise completed


In [5]:
def extract_features_rgb(images, size=96):
    features = []
    for image in tqdm(images):
        img = cv2.imread(image, cv2.IMREAD_GRAYSCALE)
        img = cv2.resize(img, (size, size))
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)  # duplicate channel to fake RGB
        features.append(img)
    features = np.array(features)
    return features

train_features = extract_features_rgb(train['image'])
test_features = extract_features_rgb(test['image'])

  0%|          | 0/28821 [00:00<?, ?it/s]

  0%|          | 0/7066 [00:00<?, ?it/s]

In [6]:
from keras.applications.mobilenet_v2 import preprocess_input

x_train = preprocess_input(train_features.astype('float32'))
x_test = preprocess_input(test_features.astype('float32'))

In [7]:
le = LabelEncoder()
le.fit(train['label'])
y_train = le.transform(train['label'])
y_test = le.transform(test['label'])
y_train = to_categorical(y_train, num_classes=7)
y_test = to_categorical(y_test, num_classes=7)

In [8]:
base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(96,96,3))
base_model.trainable = False  # freeze the backbone initially

model = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dense(256, activation='relu'),
    Dropout(0.5),
    Dense(7, activation='softmax')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


Model: "sequential"

┏━━━━━━━━━━━━┳━━━━━━━━┳━━━━━┓
┃ Layer      ┃ Output ┃ Pa… ┃
┃ (type)     ┃ Shape  ┃   # ┃
┡━━━━━━━━━━━━╇━━━━━━━━╇━━━━━┩
│ mobilenet… │ (None, │ 2,… │
│ (Function… │ 3, 3,  │     │
│            │ 1280)  │     │
├────────────┼────────┼─────┤
│ global_av… │ (None, │   0 │
│ (GlobalAv… │ 1280)  │     │
├────────────┼────────┼─────┤
│ dense      │ (None, │ 32… │
│ (Dense)    │ 256)   │     │
├────────────┼────────┼─────┤
│ dropout    │ (None, │   0 │
│ (Dropout)  │ 256)   │     │
├────────────┼────────┼─────┤
│ dense_1    │ (None, │ 1,… │
│ (Dense)    │ 7)     │     │
└────────────┴────────┴─────┘

 Total params: 2,587,719 (9.87 MB)

 Trainable params: 329,735 (1.26 MB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [9]:
y_integers = np.argmax(y_train, axis=1)
class_weights = compute_class_weight('balanced', classes=np.unique(y_integers), y=y_integers)
class_weight_dict = dict(enumerate(class_weights))

In [10]:
checkpoint = ModelCheckpoint(
    filepath="best_transfer_model.keras",
    monitor="val_loss",
    save_best_only=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1
)

early_stop = EarlyStopping(
    monitor='val_loss', patience=8, restore_best_weights=True, verbose=1
)

In [11]:
model.fit(
    x_train, y_train,
    batch_size=64,
    epochs=30,
    validation_data=(x_test, y_test),
    callbacks=[checkpoint, reduce_lr, early_stop],
    class_weight=class_weight_dict
)

KeyboardInterrupt: 

In [ ]:
base_model.trainable = True

# freeze all but the last ~20 layers
for layer in base_model.layers[:-20]:
    layer.trainable = False

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
# note: recompiling with default adam resets LR to default 0.001 — for fine-tuning,
# it's better to explicitly set a low learning rate:
from keras.optimizers import Adam
model.compile(optimizer=Adam(learning_rate=1e-5), loss='categorical_crossentropy', metrics=['accuracy'])

model.fit(
    x_train, y_train,
    batch_size=64,
    epochs=20,
    validation_data=(x_test, y_test),
    callbacks=[checkpoint, reduce_lr, early_stop],
    class_weight=class_weight_dict
)